# Generate Report Daily DS

Notebook ini mengolah **2 file data mentah**:

1. **Detail Data** (`Detail_Data_<periode>....csv`) — data harian per toko darkstore (JHK, SALES, SALES_TAGI, STD, APC, PERCENT_GM, PCT_OOS_OFMB, dst).
2. **[OOS] By Toko** (`.xlsx`) — persentase Out of Stock (%OOS TAG I, %OOS OFMB, %OOS TAG K, %OOS ALL) per toko.

Menjadi **1 file Excel akhir `Report Daily DS.xlsx`** dengan struktur sheet yang sama seperti contoh yang diberikan:

- **Data** — salinan mentah Detail Data
- **Data `<tanggal terbaru>`** & **Data `<tanggal sebelumnya>`** — ringkasan per toko untuk 1 hari tsb (dipakai untuk kolom *Daily* di Summary, dengan mekanisme *fallback*: kalau toko tidak ada datanya di tanggal terbaru, dilihat 1 hari sebelumnya)
- **Pivot Table** — ringkasan per toko untuk tanggal terbaru vs. seluruh periode (versi statis/nilai, bukan PivotTable interaktif Excel)
- **OOS** — salinan data OOS By Toko (baris "Grand Total" dibuang)
- **Summary** — laporan akhir: kolom *Daily* (formula `VLOOKUP` ke sheet `Data <tanggal>`) + kolom *Periode* (JHK, Total Net Sales, SPD, STD, APC, %GM, %OOS) diurutkan dari Total Net Sales terbesar

> **Catatan asumsi** (silakan sesuaikan di bagian KONFIGURASI bila berbeda):
> - Kolom Daily di Summary memakai *fallback* maksimal 2 hari ke belakang (persis seperti contoh: kalau toko kosong di hari terakhir, dipakai hari sebelumnya).
> - Nama sheet harian pakai singkatan bulan gaya Indonesia (`Sept`, `Okt`, `Nov`, dst) — bisa diubah di `BULAN_ABBR_ID`.
> - Sheet "Pivot Table" dibuat sebagai tabel nilai statis (hasil `groupby`), bukan objek PivotTable Excel asli, karena PivotTable interaktif tidak bisa dibuat secara andal lewat kode (openpyxl). Datanya tetap sama persis, hanya tidak bisa di-klik/filter interaktif seperti PivotTable Excel.

## 1. Install & Import Library

In [ ]:
# openpyxl & pandas biasanya sudah tersedia di Google Colab.
# Baris di bawah aman dijalankan berulang (tidak install ulang kalau sudah ada).
!pip install openpyxl pandas -q

import pandas as pd
import numpy as np
import glob, os, zipfile, shutil
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter, column_index_from_string


## 2. Upload File Input (ZIP)

Siapkan 1 file **ZIP** yang berisi:
- `Detail_Data_....csv`
- `[OOS] By Toko.xlsx`

lalu upload lewat sel di bawah (kalau dijalankan di Google Colab). Kalau dijalankan **bukan** di Colab (mis. Jupyter lokal), lewati sel upload dan isi langsung variabel `ZIP_PATH` dengan path filenya.

In [ ]:
INPUT_DIR = "extracted_input"   # folder hasil ekstrak ZIP
OUTPUT_PATH = "Report Daily DS.xlsx"

try:
    from google.colab import files
    print("Silakan upload file ZIP input (berisi Detail_Data....csv & [OOS] By Toko.xlsx):")
    uploaded = files.upload()
    ZIP_PATH = list(uploaded.keys())[0]
except ImportError:
    # bukan di Google Colab -> isi manual path ZIP di sini
    ZIP_PATH = "data_daily_perform_DS.zip"
    print(f"Bukan di Google Colab. Menggunakan ZIP_PATH = '{ZIP_PATH}' (ubah sesuai kebutuhan).")

if os.path.exists(INPUT_DIR):
    shutil.rmtree(INPUT_DIR)
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(INPUT_DIR)

print("\nIsi folder input:")
for root, _, fs in os.walk(INPUT_DIR):
    for f in fs:
        print(" -", os.path.join(root, f))


Silakan upload file ZIP input (berisi Detail_Data....csv & [OOS] By Toko.xlsx):


Saving mtd 13-9-2026.zip to mtd 13-9-2026.zip

Isi folder input:
 - extracted_input/mtd 13-9-2026/Detail_Data_13-9-2026.csv
 - extracted_input/mtd 13-9-2026/OOS By Toko.xlsx


## 3. Konfigurasi

Ubah bagian ini kalau ada penyesuaian (nama bulan, jumlah hari fallback, dsb).

In [ ]:
BULAN_ABBR_ID = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'Mei',6:'Jun',
                  7:'Jul',8:'Agu',9:'Sept',10:'Okt',11:'Nov',12:'Des'}
BULAN_FULL_ID = {1:'Januari',2:'Februari',3:'Maret',4:'April',5:'Mei',6:'Juni',
                  7:'Juli',8:'Agustus',9:'September',10:'Oktober',11:'November',12:'Desember'}

# jumlah hari fallback utk kolom Daily di Summary (default 2, sama seperti contoh)
N_FALLBACK_DAYS = 2

FILL_HEADER = PatternFill("solid", fgColor="D9E1F2")
FONT_TITLE = Font(bold=True, size=14)
FONT_SUB = Font(size=11)
FONT_HEADER = Font(bold=True, size=12)
FONT_HEADER_SM = Font(bold=True, size=11)
ALIGN_CENTER = Alignment(horizontal="center", vertical="center", wrap_text=True)
THIN = Side(style="thin", color="000000")
BORDER_ALL = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)

FMT_INT = "#,##0"
FMT_PCT = "0.00%"


## 4. Baca Data Mentah

Mencari otomatis file **Detail Data (.csv)** dan **[OOS] By Toko (.xlsx)** di dalam folder hasil ekstrak, lalu membacanya.

In [ ]:
def find_input_files(input_dir):
    """Cari file Detail Data (csv) dan file OOS By Toko (xlsx) di dalam folder input."""
    csv_candidates = glob.glob(os.path.join(input_dir, "**", "*Detail*Data*.csv"), recursive=True)
    if not csv_candidates:
        csv_candidates = glob.glob(os.path.join(input_dir, "**", "*.csv"), recursive=True)

    oos_candidates = [f for f in glob.glob(os.path.join(input_dir, "**", "*.xlsx"), recursive=True)
                      if "oos" in os.path.basename(f).lower() and "report daily" not in os.path.basename(f).lower()]
    if not oos_candidates:
        oos_candidates = [f for f in glob.glob(os.path.join(input_dir, "**", "*.xlsx"), recursive=True)
                          if "report daily" not in os.path.basename(f).lower()]

    if not csv_candidates:
        raise FileNotFoundError("File Detail Data (.csv) tidak ditemukan di dalam ZIP input.")
    if not oos_candidates:
        raise FileNotFoundError("File [OOS] By Toko (.xlsx) tidak ditemukan di dalam ZIP input.")
    return csv_candidates[0], oos_candidates[0]


def load_detail_data(csv_path):
    df = pd.read_csv(csv_path)
    df["TANGGAL"] = pd.to_datetime(df["TANGGAL"])
    return df


def load_oos_data(xlsx_path):
    df = pd.read_excel(xlsx_path, sheet_name=0)
    df.columns = [c.strip() for c in df.columns]
    # buang baris "Grand Total"
    df = df[df.iloc[:, 0].astype(str).str.strip().str.lower() != "grand total"]
    df = df.dropna(subset=[df.columns[0]]).reset_index(drop=True)
    return df


csv_path, oos_path = find_input_files(INPUT_DIR)
print("Detail Data :", csv_path)
print("OOS By Toko :", oos_path)

df_detail = load_detail_data(csv_path)
df_oos = load_oos_data(oos_path)

print(f"\nDetail Data : {df_detail.shape[0]} baris, {df_detail['KD_STORE'].nunique()} toko, periode {df_detail['TANGGAL'].min().date()} s.d. {df_detail['TANGGAL'].max().date()}")
print(f"OOS By Toko : {df_oos.shape[0]} toko")


Detail Data : extracted_input/mtd 13-9-2026/Detail_Data_13-9-2026.csv
OOS By Toko : extracted_input/mtd 13-9-2026/OOS By Toko.xlsx

Detail Data : 866 baris, 67 toko, periode 2026-09-01 s.d. 2026-09-13
OOS By Toko : 68 toko


## 5. Fungsi Agregasi

- `sort_by_net_sales(g)` → helper pengurut: **Total Net Sales terbesar → terkecil**
- `daily_summary(df, tanggal)` → ringkasan per toko untuk **1 tanggal** (sheet `Data <tanggal>`), sudah tersortir
- `daily_summary_full(df, tanggal)` / `period_summary(df)` → dipakai sheet `Pivot Table` & kolom periode `Summary`, sudah tersortir

> Semua sheet level toko (`Data <tanggal>`, `Pivot Table`, `Summary`) kini memakai urutan yang sama: Net Sales terbesar di baris paling atas.

In [ ]:
# ---------------------------------------------------------------
# SORT: semua tabel per toko diurutkan Total Net Sales TERBESAR -> TERKECIL
# ---------------------------------------------------------------
def sort_by_net_sales(g, sales_col="Sum of SALES"):
    """Urutkan tabel per toko dari Total Net Sales terbesar ke terkecil."""
    if sales_col not in g.columns:
        return g.reset_index(drop=True)
    return (g.sort_values(sales_col, ascending=False, na_position="last")
             .reset_index(drop=True))


def daily_summary(df, tanggal):
    """Ringkasan per toko untuk SATU tanggal tertentu (setara sheet 'Data <tanggal>').

    Diurutkan Total Net Sales (SALES) terbesar -> terkecil. Kolom bantu '__sales__'
    hanya dipakai untuk sorting lalu dibuang, supaya layout kolom A:I tetap sama
    (VLOOKUP di sheet Summary tidak berubah).
    """
    sub = df[df["TANGGAL"] == tanggal]
    g = sub.groupby(["KD_STORE", "NAMA_STORE", "NAMA_BRANCH"], as_index=False).agg(
        **{
            "Sum of JHK": ("JHK", "sum"),
            "Average of SPD": ("SPD", "mean"),
            "SPD Tag I": ("SALES_TAGI", "sum"),
            "Average of STD": ("STD", "mean"),
            "Average of APC": ("APC", "mean"),
            "Average of PERCENT_GM": ("PERCENT_GM", "mean"),
            "__sales__": ("SALES", "sum"),
        }
    )
    g = (g.sort_values("__sales__", ascending=False, na_position="last")
           .drop(columns="__sales__")
           .reset_index(drop=True))
    return g


def daily_summary_full(df, tanggal):
    """Sama seperti daily_summary tapi dengan kolom lengkap ala blok 'Pivot Table'
    (Sum SALES, Sum SALES_TAGI, Average PCT_OOS_OFMB ikut disertakan).
    Diurutkan Sum of SALES terbesar -> terkecil."""
    sub = df[df["TANGGAL"] == tanggal]
    g = sub.groupby(["KD_STORE", "NAMA_STORE", "NAMA_BRANCH"], as_index=False).agg(
        **{
            "Sum of JHK": ("JHK", "sum"),
            "Sum of SALES": ("SALES", "sum"),
            "Sum of SALES_TAGI": ("SALES_TAGI", "sum"),
            "Average of SPD": ("SPD", "mean"),
            "Average of STD": ("STD", "mean"),
            "Average of APC": ("APC", "mean"),
            "Average of PERCENT_GM": ("PERCENT_GM", "mean"),
            "Average of PCT_OOS_OFMB": ("PCT_OOS_OFMB", "mean"),
        }
    )
    return sort_by_net_sales(g)


def period_summary(df):
    """Ringkasan per toko untuk SELURUH periode (setara blok 'All' pada sheet Pivot Table).
    Diurutkan Sum of SALES terbesar -> terkecil."""
    g = df.groupby(["KD_STORE", "NAMA_STORE", "NAMA_BRANCH"], as_index=False).agg(
        **{
            "Sum of JHK": ("JHK", "sum"),
            "Sum of SALES": ("SALES", "sum"),
            "Sum of SALES_TAGI": ("SALES_TAGI", "sum"),
            "Average of SPD": ("SPD", "mean"),
            "Average of STD": ("STD", "mean"),
            "Average of APC": ("APC", "mean"),
            "Average of PERCENT_GM": ("PERCENT_GM", "mean"),
            "Average of PCT_OOS_OFMB": ("PCT_OOS_OFMB", "mean"),
        }
    )
    return sort_by_net_sales(g)


def sheet_label(tanggal):
    ts = pd.Timestamp(tanggal)
    return f"Data {ts.day} {BULAN_ABBR_ID[ts.month]} {ts.year}"


def periode_title(start, end):
    start, end = pd.Timestamp(start), pd.Timestamp(end)
    if (start.year, start.month) == (end.year, end.month):
        return f"Periode {start.day} - {end.day} {BULAN_FULL_ID[start.month]} {start.year}"
    return (f"Periode {start.day} {BULAN_FULL_ID[start.month]} {start.year} - "
            f"{end.day} {BULAN_FULL_ID[end.month]} {end.year}")


## 6. Bangun Tabel `Summary` (level toko)

Untuk setiap toko:
- Kolom **Daily** (F-K) → ditentukan sheet `Data <tanggal>` mana yang dipakai (tanggal terbaru; kalau toko tidak ada datanya di situ, mundur ke tanggal sebelumnya — *fallback* sebanyak `N_FALLBACK_DAYS`). Nilainya sendiri ditulis sebagai **formula VLOOKUP** ke sheet tsb saat proses penulisan Excel, bukan nilai statis, supaya kalau data harian direvisi, Summary otomatis ikut berubah.
- Kolom **Periode** (L-V) → dihitung dari seluruh periode (`period_summary`) + %OOS dari file OOS By Toko.

In [ ]:
def build_summary_table(df_detail, df_oos, n_fallback_days=N_FALLBACK_DAYS):
    all_dates = sorted(df_detail["TANGGAL"].unique(), reverse=True)
    fallback_dates = all_dates[:n_fallback_days]

    # tabel harian per tanggal fallback -> dipakai juga untuk sheet 'Data <tanggal>'
    daily_tables = {pd.Timestamp(d): daily_summary(df_detail, d) for d in fallback_dates}

    # ringkasan penuh 1 periode (semua tanggal)
    df_period = period_summary(df_detail)

    # daftar master toko (ambil nama/cabang paling akhir muncul)
    master = (df_detail.sort_values("TANGGAL")
              .drop_duplicates(subset="KD_STORE", keep="last")
              [["KD_STORE", "NAMA_STORE", "NAMA_BRANCH"]])

    summary = master.merge(df_period, on=["KD_STORE", "NAMA_STORE", "NAMA_BRANCH"], how="left")

    # tentukan sheet 'Data <tanggal>' mana yang dipakai per toko (fallback)
    source_sheet, missing_stores = [], []
    for kd in summary["KD_STORE"]:
        found = False
        for d in fallback_dates:
            tbl = daily_tables[pd.Timestamp(d)]
            if (tbl["KD_STORE"] == kd).any():
                source_sheet.append(sheet_label(d))
                found = True
                break
        if not found:
            source_sheet.append(None)
            missing_stores.append(kd)
    summary["__source_sheet__"] = source_sheet

    # gabungkan %OOS dari file OOS By Toko
    oos = df_oos.rename(columns={df_oos.columns[0]: "Kode Toko"})
    oos_small = oos[["Kode Toko", "% OOS OFMB", "% OOS TAG I", "% OOS TAG K"]].copy()
    summary = summary.merge(oos_small, left_on="KD_STORE", right_on="Kode Toko", how="left")

    # urutkan besar -> kecil berdasarkan Total Net Sales periode (kolom M di Summary)
    # na_position="last" -> toko tanpa nilai SALES ditaruh paling bawah, bukan paling atas
    summary = sort_by_net_sales(summary)
    summary.insert(0, "No", range(1, len(summary) + 1))

    return summary, daily_tables, df_period, missing_stores, fallback_dates


summary, daily_tables, df_period, missing_stores, fallback_dates = build_summary_table(df_detail, df_oos)

print(f"Total toko di Summary : {len(summary)}")
print(f"Tanggal fallback yang dipakai untuk kolom Daily : {[pd.Timestamp(d).date() for d in fallback_dates]}")
if missing_stores:
    print(f"[Peringatan] {len(missing_stores)} toko tidak ada data di {N_FALLBACK_DAYS} tanggal "
          f"terakhir, kolom Daily-nya akan dikosongkan: {missing_stores}")

# cek urutan: Total Net Sales harus menurun (terbesar -> terkecil)
_s = summary["Sum of SALES"].dropna()
print("Urutan Total Net Sales menurun :", bool(_s.is_monotonic_decreasing))
print("\n5 toko Net Sales tertinggi:")
print(summary[["No", "KD_STORE", "NAMA_STORE", "Sum of SALES"]].head())
print("\n5 toko Net Sales terendah:")
print(summary[["No", "KD_STORE", "NAMA_STORE", "Sum of SALES"]].tail())
summary.head()


Total toko di Summary : 67
Tanggal fallback yang dipakai untuk kolom Daily : [datetime.date(2026, 9, 13), datetime.date(2026, 9, 12)]
Urutan Total Net Sales menurun : True

5 toko Net Sales tertinggi:
   No KD_STORE        NAMA_STORE  Sum of SALES
0   1     UF32   DS DHARMAHUSADA  9.604357e+08
1   2     R899  DS UJUNG PANDANG  9.373390e+08
2   3     UF29  DS TENGGILIS SBY  7.902022e+08
3   4     XC36        DS CIBUBUR  7.891189e+08
4   5     CI99  DS HARAPAN INDAH  7.545422e+08

5 toko Net Sales terendah:
    No KD_STORE       NAMA_STORE  Sum of SALES
62  63     W858      DS RINGROAD  4.119464e+08
63  64     1M6U        DS KEMANG  3.896985e+08
64  65     BJ57      DS ANTAPANI  3.743958e+08
65  66     TH06  DS MUARA KARANG  3.280208e+08
66  67     UF27    DS BALIKPAPAN  3.180543e+08


,No,KD_STORE,NAMA_STORE,NAMA_BRANCH,Sum of JHK,Sum of SALES,Sum of SALES_TAGI,Average of SPD,Average of STD,Average of APC,Average of PERCENT_GM,Average of PCT_OOS_OFMB,__source_sheet__,Kode Toko,% OOS OFMB,% OOS TAG I,% OOS TAG K
0,1,UF32,DS DHARMAHUSADA,SIDOARJO,13,9.604357e+08,2.005193e+08,7.387967e+07,886.769231,82547.938370,0.176165,0.014468,Data 13 Sept 2026,UF32,0.014191,0.016237,0.026484
1,2,R899,DS UJUNG PANDANG,MAKASSAR,13,9.373390e+08,1.801136e+08,7.210300e+07,656.153846,112484.936515,0.163645,0.037289,Data 13 Sept 2026,R899,0.035650,0.053349,0.028843
2,3,UF29,DS TENGGILIS SBY,SIDOARJO,13,7.902022e+08,1.976904e+08,6.078479e+07,695.846154,84962.973722,0.172440,0.013279,Data 13 Sept 2026,UF29,0.013525,0.017386,0.022811
3,4,XC36,DS CIBUBUR,CILEUNGSI_2,12,7.891189e+08,1.940637e+08,6.575991e+07,556.500000,117884.816667,0.168005,0.015058,Data 12 Sept 2026,XC36,0.015199,0.044921,0.018894
4,5,CI99,DS HARAPAN INDAH,CILEUNGSI_2,13,7.545422e+08,1.862897e+08,5.804171e+07,506.230769,115381.993612,0.162963,0.024086,Data 13 Sept 2026,CI99,0.024826,0.043546,0.024667


## 7. Fungsi Bantu Penulisan Excel (styling)

In [ ]:
def style_header_row(ws, row, col_start, col_end, font=FONT_HEADER_SM):
    for c in range(col_start, col_end + 1):
        cell = ws.cell(row=row, column=c)
        cell.font = font
        cell.fill = FILL_HEADER
        cell.alignment = ALIGN_CENTER
        cell.border = BORDER_ALL


def autosize(ws, ncols, min_width=10, max_width=32):
    for c in range(1, ncols + 1):
        letter = get_column_letter(c)
        best = min_width
        for cell in ws[letter]:
            if cell.value is not None:
                best = max(best, min(max_width, len(str(cell.value)) + 2))
        ws.column_dimensions[letter].width = best


## 8. Sheet `Data` (salinan mentah Detail Data)

In [ ]:
def write_data_sheet(wb, df_detail):
    ws = wb.create_sheet("Data")
    ws.append(list(df_detail.columns))
    style_header_row(ws, 1, 1, len(df_detail.columns))
    for row in df_detail.itertuples(index=False):
        ws.append(list(row))
    date_col = list(df_detail.columns).index("TANGGAL") + 1
    for r in range(2, ws.max_row + 1):
        ws.cell(row=r, column=date_col).number_format = "m/d/yy h:mm"
    ws.freeze_panes = "A2"
    autosize(ws, len(df_detail.columns))
    return ws


## 9. Sheet `Data <tanggal>` (ringkasan per toko, 1 hari)

In [ ]:
def write_daily_tables(wb, daily_tables):
    sheets = {}
    for tanggal, tbl in daily_tables.items():
        name = sheet_label(tanggal)
        ws = wb.create_sheet(name)
        ws.append(list(tbl.columns))
        style_header_row(ws, 1, 1, len(tbl.columns))
        for row in tbl.itertuples(index=False):
            ws.append(list(row))
        for r in range(2, ws.max_row + 1):
            ws.cell(row=r, column=5).number_format = FMT_INT   # Average of SPD
            ws.cell(row=r, column=6).number_format = FMT_INT   # SPD Tag I
            ws.cell(row=r, column=8).number_format = FMT_INT   # Average of APC
            ws.cell(row=r, column=9).number_format = FMT_PCT   # Average of PERCENT_GM
        autosize(ws, len(tbl.columns))
        sheets[tanggal] = ws
    return sheets


## 10. Sheet `Pivot Table` (ringkasan tanggal terbaru vs. seluruh periode)

> Sheet ini dibuat sebagai **tabel nilai statis** (hasil `groupby`), bukan objek PivotTable Excel asli — datanya identik, hanya tidak bisa di-klik/filter interaktif seperti PivotTable bawaan Excel.

In [ ]:
def write_pivot_sheet(wb, df_period, df_detail, latest_date):
    ws = wb.create_sheet("Pivot Table")
    left = daily_summary_full(df_detail, latest_date)
    right = df_period

    ws.cell(row=1, column=1, value=f"Filter Tanggal: {pd.Timestamp(latest_date).strftime('%d/%m/%Y')}").font = Font(bold=True)
    headers = list(left.columns)
    ncol_left = len(headers)
    for j, h in enumerate(headers, start=1):
        ws.cell(row=2, column=j, value=h)
    style_header_row(ws, 2, 1, ncol_left)
    for i, row in enumerate(left.itertuples(index=False), start=3):
        for j, v in enumerate(row, start=1):
            ws.cell(row=i, column=j, value=v)

    gap = 2
    start_col_right = ncol_left + gap + 1
    ws.cell(row=1, column=start_col_right, value="Filter Tanggal: (All)").font = Font(bold=True)
    for j, h in enumerate(headers, start=start_col_right):
        ws.cell(row=2, column=j, value=h)
    style_header_row(ws, 2, start_col_right, start_col_right + ncol_left - 1)
    for i, row in enumerate(right.itertuples(index=False), start=3):
        for j, v in enumerate(row, start=start_col_right):
            ws.cell(row=i, column=j, value=v)

    for block_start in (1, start_col_right):
        for r in range(3, 3 + max(len(left), len(right))):
            for j, h in enumerate(headers, start=block_start):
                if h.startswith("Sum of SALES") or h.startswith("Average of SPD") \
                   or h.startswith("Average of STD") or h.startswith("Average of APC"):
                    ws.cell(row=r, column=j).number_format = FMT_INT
                elif h.startswith("Average of PERCENT_GM") or h.startswith("Average of PCT_OOS"):
                    ws.cell(row=r, column=j).number_format = FMT_PCT

    autosize(ws, start_col_right + ncol_left - 1)
    return ws


## 11. Sheet `OOS` (salinan data OOS By Toko, tanpa baris Grand Total)

In [ ]:
def write_oos_sheet(wb, df_oos):
    ws = wb.create_sheet("OOS")
    ws.append(list(df_oos.columns))
    style_header_row(ws, 1, 1, len(df_oos.columns))
    for row in df_oos.itertuples(index=False):
        ws.append(list(row))
    for r in range(2, ws.max_row + 1):
        for c in range(4, len(df_oos.columns) + 1):
            ws.cell(row=r, column=c).number_format = FMT_PCT
    autosize(ws, len(df_oos.columns))
    return ws


## 12. Sheet `Summary` (laporan akhir)

- Kolom **Daily** (F-K) ditulis sebagai **formula `VLOOKUP`** ke sheet `Data <tanggal>` yang sesuai per toko (mengikuti hasil *fallback* di langkah 6).
- Kolom **Periode** (L-V) ditulis sebagai nilai (JHK, Total Net Sales, Total Net Sales Tag I, Average SPD, STD, APC, %GM) + formula `=N/L` untuk SPD Tag I, + %OOS dari file OOS By Toko.
- Baris terakhir **Average/Sum** memakai formula `AVERAGE`/`SUM`, jadi otomatis mengikuti kalau ada baris toko yang ditambah/dikurangi.

In [ ]:
def write_summary_sheet(wb, summary, start_date, end_date):
    ws = wb.create_sheet("Summary")

    ws.cell(row=3, column=2, value="Daily Report Performance Darkstore").font = FONT_TITLE
    ws.cell(row=4, column=2, value=periode_title(start_date, end_date)).font = FONT_SUB

    end_ts = pd.Timestamp(end_date)
    label_daily = f"Sales Darkstore {end_ts.day} {BULAN_FULL_ID[end_ts.month]} {end_ts.year}"
    label_period = f"Sales Darkstore {periode_title(start_date, end_date).replace('Periode ', '')}"

    header_row1, header_row2 = 5, 6
    ws.cell(row=header_row1, column=2, value="No")
    ws.cell(row=header_row1, column=3, value="Kode Toko")
    ws.cell(row=header_row1, column=4, value="Nama Toko")
    ws.cell(row=header_row1, column=5, value="Cabang")
    ws.cell(row=header_row1, column=6, value=label_daily)
    ws.cell(row=header_row1, column=12, value=label_period)
    ws.merge_cells(start_row=header_row1, start_column=2, end_row=header_row2, end_column=2)
    ws.merge_cells(start_row=header_row1, start_column=3, end_row=header_row2, end_column=3)
    ws.merge_cells(start_row=header_row1, start_column=4, end_row=header_row2, end_column=4)
    ws.merge_cells(start_row=header_row1, start_column=5, end_row=header_row2, end_column=5)
    ws.merge_cells(start_row=header_row1, start_column=6, end_row=header_row1, end_column=11)
    ws.merge_cells(start_row=header_row1, start_column=12, end_row=header_row1, end_column=22)

    sub_headers = [" JHK", "SPD", "SPD Tag I", "STD", "APC", "%GM",
                   "JHK", "Total Net Sales", "Total Net Sales Tag I", "SPD", "SPD Tag I",
                   "STD", "APC", "%GM", "%OOS OFMB", "%OOS I", "%OOS K"]
    for j, h in enumerate(sub_headers, start=6):
        ws.cell(row=header_row2, column=j, value=h)

    style_header_row(ws, header_row1, 2, 22, font=FONT_HEADER)
    style_header_row(ws, header_row2, 2, 22, font=FONT_HEADER)

    first_data_row = 7
    n = len(summary)
    records = summary.to_dict("records")
    for i, d in enumerate(records, start=first_data_row):
        r = i
        ws.cell(row=r, column=2, value=d["No"])
        ws.cell(row=r, column=3, value=d["KD_STORE"])
        ws.cell(row=r, column=4, value=d["NAMA_STORE"])
        ws.cell(row=r, column=5, value=d["NAMA_BRANCH"])

        src = d["__source_sheet__"]
        if src is not None:
            ws.cell(row=r, column=6, value=f"=VLOOKUP(C{r},'{src}'!A:I, 4,FALSE)")
            ws.cell(row=r, column=7, value=f"=VLOOKUP(C{r},'{src}'!A:I, 5,FALSE)")
            ws.cell(row=r, column=8, value=f"=VLOOKUP(C{r},'{src}'!A:I, 6,FALSE)")
            ws.cell(row=r, column=9, value=f"=VLOOKUP(C{r},'{src}'!A:I, 7,FALSE)")
            ws.cell(row=r, column=10, value=f"=VLOOKUP(C{r},'{src}'!A:I, 8,FALSE)")
            ws.cell(row=r, column=11, value=f"=VLOOKUP(C{r},'{src}'!A:I, 9,FALSE)")
        else:
            for c in range(6, 12):
                ws.cell(row=r, column=c, value=None)

        ws.cell(row=r, column=12, value=d["Sum of JHK"])
        ws.cell(row=r, column=13, value=d["Sum of SALES"])
        ws.cell(row=r, column=14, value=d["Sum of SALES_TAGI"])
        ws.cell(row=r, column=15, value=d["Average of SPD"])
        ws.cell(row=r, column=16, value=f"=N{r}/L{r}")
        ws.cell(row=r, column=17, value=d["Average of STD"])
        ws.cell(row=r, column=18, value=d["Average of APC"])
        ws.cell(row=r, column=19, value=d["Average of PERCENT_GM"])
        ws.cell(row=r, column=20, value=d.get("% OOS OFMB"))
        ws.cell(row=r, column=21, value=d.get("% OOS TAG I"))
        ws.cell(row=r, column=22, value=d.get("% OOS TAG K"))

        for c in range(2, 23):
            ws.cell(row=r, column=c).border = BORDER_ALL
            ws.cell(row=r, column=c).alignment = Alignment(horizontal="center")
        for c in (7, 8, 9, 10, 13, 14, 15, 16, 17, 18):
            ws.cell(row=r, column=c).number_format = FMT_INT
        for c in (11, 19, 20, 21, 22):
            ws.cell(row=r, column=c).number_format = FMT_PCT
        ws.cell(row=r, column=6).number_format = FMT_INT
        ws.cell(row=r, column=12).number_format = "0"

    total_row = first_data_row + n
    ws.cell(row=total_row, column=2, value="Average/Sum ")
    ws.merge_cells(start_row=total_row, start_column=2, end_row=total_row, end_column=5)
    for col_letter in ["F", "G", "H", "I", "J", "K", "L", "O", "P", "Q", "R", "S", "T", "U", "V"]:
        c = column_index_from_string(col_letter)
        ws.cell(row=total_row, column=c, value=f"=AVERAGE({col_letter}{first_data_row}:{col_letter}{total_row-1})")
    for col_letter in ["M", "N"]:
        c = column_index_from_string(col_letter)
        ws.cell(row=total_row, column=c, value=f"=SUM({col_letter}{first_data_row}:{col_letter}{total_row-1})")
    for c in range(2, 23):
        ws.cell(row=total_row, column=c).font = Font(bold=True)
        ws.cell(row=total_row, column=c).border = BORDER_ALL

    ws.freeze_panes = ws.cell(row=first_data_row, column=1).coordinate
    autosize(ws, 22)
    ws.column_dimensions["B"].width = 6
    return ws


## 13. Jalankan Semua & Simpan Workbook

Membuat workbook baru, menulis semua sheet dengan urutan yang sama seperti contoh (`Data` → `Data <tgl-2>` → `Data <tgl-1>` → `Pivot Table` → `OOS` → `Summary`), lalu menyimpannya.

In [ ]:
wb = openpyxl.Workbook()
wb.remove(wb.active)

write_data_sheet(wb, df_detail)
write_daily_tables(wb, daily_tables)
write_pivot_sheet(wb, df_period, df_detail, latest_date=fallback_dates[0])
write_oos_sheet(wb, df_oos)
write_summary_sheet(wb, summary, df_detail["TANGGAL"].min(), df_detail["TANGGAL"].max())

# urutan sheet: Data, Data <tgl lama>, Data <tgl baru>, Pivot Table, OOS, Summary
order = ["Data"] + [sheet_label(d) for d in sorted(fallback_dates)] + ["Pivot Table", "OOS", "Summary"]
wb._sheets = [wb[name] for name in order]

wb.save(OUTPUT_PATH)
print(f"Tersimpan: {OUTPUT_PATH}")
print(f"Sheet: {wb.sheetnames}")


Tersimpan: Report Daily DS.xlsx
Sheet: ['Data', 'Data 12 Sept 2026', 'Data 13 Sept 2026', 'Pivot Table', 'OOS', 'Summary']


## 14. Hitung Ulang Formula (Recalculate)

Karena formula (`VLOOKUP`, `AVERAGE`, `SUM`) ditulis lewat `openpyxl`, nilainya belum ter-cache — Excel/LibreOffice perlu membuka & menghitung ulang file sekali supaya semua sel formula punya nilai saat dibuka nanti. Kalau notebook ini dijalankan di **Google Colab**, install LibreOffice dulu (sekali saja per sesi).

In [ ]:
# Install LibreOffice (headless) kalau belum ada -- dipakai untuk recalculate formula.
# Di Google Colab biasanya perlu diinstall dulu (~1-2 menit, sekali per sesi).
import shutil as _shutil
if _shutil.which("soffice") is None:
    print("Menginstall LibreOffice (sekali saja per sesi Colab)...")
    !apt-get -qq update && apt-get -qq install -y libreoffice-calc > /dev/null
else:
    print("LibreOffice sudah tersedia.")


Menginstall LibreOffice (sekali saja per sesi Colab)...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
import subprocess, tempfile

def recalc_with_libreoffice(path, timeout=90):
    """Buka file dengan LibreOffice (headless) lalu hitung ulang semua formula
    supaya nilainya ter-cache. Dikonversi ke FOLDER SEMENTARA (bukan overwrite
    langsung di tempat) karena beberapa environment gagal/silent-error saat
    source dan tujuan konversi adalah file yang sama persis.
    Kalau LibreOffice tidak tersedia, langkah ini dilewati -- file tetap valid,
    hanya saja sel formula akan terlihat kosong sampai dibuka & disimpan
    sekali lewat Excel/LibreOffice.
    """
    soffice = _shutil.which("soffice") or _shutil.which("libreoffice")
    if soffice is None:
        print("LibreOffice tidak ditemukan -- lewati recalculate. "
              "Buka & simpan ulang file ini sekali lewat Excel supaya formula ter-cache.")
        return False

    abs_path = os.path.abspath(path)
    with tempfile.TemporaryDirectory() as tmpdir:
        cmd = [soffice, "--headless", "--calc", "--convert-to", "xlsx", "--outdir", tmpdir, abs_path]
        try:
            result = subprocess.run(cmd, timeout=timeout, capture_output=True, text=True)
            converted = os.path.join(tmpdir, os.path.basename(abs_path))
            if result.returncode == 0 and os.path.exists(converted) and os.path.getsize(converted) > 0:
                shutil.move(converted, abs_path)
                print("Recalculate selesai, semua formula sudah ter-cache nilainya.")
                return True
            print(f"Recalculate gagal (returncode={result.returncode}): {result.stderr.strip()[:300]}\n"
                  "-- file tetap tersimpan, hanya formula belum ter-cache.")
            return False
        except Exception as e:
            print(f"Recalculate gagal ({e}) -- file tetap tersimpan, hanya formula belum ter-cache.")
            return False

recalc_with_libreoffice(OUTPUT_PATH)


Recalculate selesai, semua formula sudah ter-cache nilainya.


True

## 15. Download Hasil

In [ ]:
try:
    from google.colab import files
    files.download(OUTPUT_PATH)
except ImportError:
    print(f"File hasil ada di: {os.path.abspath(OUTPUT_PATH)}")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>